# GSM8K Ablation — Full Sweep, Staged (512 → 768 tokens, repetition_penalty=1.0)

Standalone follow-up to `gsm8k-evaluation.ipynb`. That notebook showed exact-match is
contaminated by two stacked effects: real weight-level interference from merging, and a
decoding-config artifact (`repetition_penalty=1.3` + `max_new_tokens=320` truncating CoT before
`#### <answer>` is ever emitted — hit rates as high as 155/200 samples for the SLERP orderings).

This notebook reruns **all 14 models** (5 merge methods + 6 SLERP orderings + 3 specialists)
staged as follows:

| Setting              |           Old |           Stage 1 |         Stage 2 (conditional) |
|-----------------------|--------------:|-------------------:|-------------------------------:|
| `max_new_tokens`      |           320 |                512 |                             768 |
| `repetition_penalty`  |           1.3 |                1.0 |                              1.0 |
| `temperature`/`top_p` |        (same) |             (same) |                          (same) |

**Stage 1** reruns every model at 512 tokens. **Stage 2** only reruns models whose stage-1
cap-hit rate is still high (>10% of samples, i.e. >20/200) — those are the ones where 512 may
still be truncating a genuinely longer reasoning chain. Going straight to 768/1024 for everything
wastes generation time on models that already terminate cleanly well under 512.

For every model this records: EM, mean generated length (tokens), cap-hit rate, and extraction-path
breakdown. Decoding-independent teacher-forced NLL is not recomputed — it doesn't depend on these
settings and the original notebook's numbers are already valid; merge them in from
`gsm8k_diagnostic_results.pkl` if you want NLL alongside this table.

**Reading the result per model:**
- Cap-hit rate drops and EM rises → decoding budget was the limiting factor.
- Cap-hit rate is already low (even in the old 320-token run) but EM stays poor → the merged
  weights themselves are responsible, not the budget.

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

In [2]:
import torch
import gc
import re
import pickle
from collections import Counter, defaultdict
from datasets import load_dataset
from tqdm import tqdm
from unsloth import FastLanguageModel

# ─────────────────────────────────────────────
# SHARED: identical to gsm8k-evaluation.ipynb, kept verbatim so results
# are directly comparable at the extraction-logic level.
# ─────────────────────────────────────────────

def build_chat_prompt(tokenizer, user_content: str) -> str:
    messages = [{"role": "user", "content": user_content}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,  # Qwen3 <think> block otherwise eats the whole budget
    )


def prep_tokenizer_for_generation(tokenizer):
    """Left-padding is required for correct batched causal-LM generation."""
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def format_gsm8k_prompt(question: str) -> str:
    return (
        f"Solve the following math problem. Show your reasoning and put "
        f"your final numeric answer after \'#### \'.\n\n"
        f"Question: {question}"
    )

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Diagnostic exact-match eval

Same `extract_gsm8k_answer_diagnostic` / `evaluate_gsm8k_diagnostic` as the original notebook,
plus **mean generated length** per model (needed to judge whether 512 vs. 768 is the right call,
not just the binary cap-hit flag).

In [3]:
def extract_gsm8k_answer_diagnostic(text: str):
    """Extract final answer — tries \'#### X\' first, then \\boxed{}, then last
    number anywhere in the text. Returns (answer, path) where path is one of
    \'hash\', \'boxed\', \'fallback_last_number\', \'no_number_found\'."""
    hash_match = re.findall(r"####\s*(-?[\d,]+\.?\d*)", text)
    if hash_match:
        return hash_match[-1].replace(",", "").strip(), "hash"
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed[-1].replace(",", "").strip(), "boxed"
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "").strip(), "fallback_last_number"
    return text.strip(), "no_number_found"


def extract_gsm8k_answer(text: str) -> str:
    ans, _ = extract_gsm8k_answer_diagnostic(text)
    return ans


def evaluate_gsm8k_diagnostic(
    model,
    tokenizer,
    model_name: str = "model",
    num_samples: int = 200,
    batch_size: int = 4,
    max_new_tokens: int = 320,
    repetition_penalty: float = 1.3,
    device: str = "cuda",
) -> dict:
    """Evaluate on GSM8K (test split) using exact match, with per-sample
    attribution of which extraction path fired, generated length in tokens,
    and whether generation hit the max_new_tokens cap without emitting EOS."""
    print(f"\n{'─'*60}")
    print(f"[GSM8K-diagnostic] Evaluating: {model_name}  (max_new_tokens={max_new_tokens}, repetition_penalty={repetition_penalty})")
    print(f"{'─'*60}")

    model.eval()
    prep_tokenizer_for_generation(tokenizer)

    dataset = load_dataset("gsm8k", "main", split="test")
    dataset = dataset.select(range(min(num_samples, len(dataset))))

    records = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [gsm8k-diag]"):
        batch = dataset[i : i + batch_size]

        questions    = batch["question"]
        true_answers = [extract_gsm8k_answer(a) for a in batch["answer"]]
        prompts      = [build_chat_prompt(tokenizer, format_gsm8k_prompt(q)) for q in questions]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
            add_special_tokens=False,  # chat template already adds special tokens
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=repetition_penalty,
            )

        input_len = inputs["input_ids"].shape[1]
        for j, output in enumerate(outputs):
            gen_ids     = output[input_len:]
            generated   = tokenizer.decode(gen_ids, skip_special_tokens=True)
            pred_answer, path = extract_gsm8k_answer_diagnostic(generated)

            nonpad_ids  = gen_ids[gen_ids != tokenizer.pad_token_id]
            gen_len     = len(nonpad_ids)
            hit_len_cap = (
                gen_len >= max_new_tokens
                and (gen_len == 0 or nonpad_ids[-1].item() != tokenizer.eos_token_id)
            )

            records.append({
                "idx"         : i + j,
                "question"    : questions[j],
                "true_answer" : true_answers[j],
                "pred_answer" : pred_answer,
                "extraction_path": path,
                "gen_len"     : gen_len,
                "hit_len_cap" : bool(hit_len_cap),
                "correct"     : int(pred_answer.strip() == true_answers[j].strip()),
                "generated"   : generated,
            })

    per_sample_exact = [r["correct"] for r in records]
    exact_match      = round(sum(per_sample_exact) / len(per_sample_exact), 4)

    path_counts  = Counter(r["extraction_path"] for r in records)
    path_correct = defaultdict(int)
    for r in records:
        path_correct[r["extraction_path"]] += r["correct"]

    breakdown = {
        path: {
            "n"       : n,
            "correct" : path_correct[path],
            "acc"     : round(path_correct[path] / n, 4) if n else 0.0,
        }
        for path, n in path_counts.items()
    }

    len_cap_hits = sum(r["hit_len_cap"] for r in records)
    mean_gen_len = round(sum(r["gen_len"] for r in records) / len(records), 1)

    result = {
        "repo_id"          : model_name,
        "exact_match"      : exact_match,
        "num_samples"      : len(records),
        "records"          : records,
        "extraction_breakdown": breakdown,
        "len_cap_hits"     : len_cap_hits,
        "cap_hit_rate"     : round(len_cap_hits / len(records), 4),
        "mean_gen_len"     : mean_gen_len,
        "max_new_tokens"   : max_new_tokens,
    }

    print(f"  Exact Match: {exact_match:.4f}  |  mean gen length: {mean_gen_len}  |  len-cap hit on {len_cap_hits}/{len(records)} ({result['cap_hit_rate']:.1%})")
    print(f"  {'path':<22} {'n':>5} {'correct':>8} {'acc':>7}")
    for path, d in sorted(breakdown.items(), key=lambda kv: -kv[1]['n']):
        print(f"  {path:<22} {d['n']:>5} {d['correct']:>8} {d['acc']:>7.3f}")

    return result

## Full model list + original (320-token) reference values

Identical repo list to `gsm8k-evaluation.ipynb`, cell 10. `ORIGINAL_EM` / `ORIGINAL_LEN_CAP_HITS`
are hardcoded from that notebook's `repetition_penalty=1.3, max_new_tokens=320` run, used as the
comparison baseline. If `gsm8k_diagnostic_results.pkl` is present, its numbers are used instead.

In [4]:
import itertools

adapter_names = ["dolly", "metamath", "codealpaca"]

slerp_order_repos = [
    f"Srishtik/Qwen3-0.6B-slerp-order-{'-'.join(perm)}-3-adapters-merged-2"
    for perm in itertools.permutations(adapter_names)
]

merged_repos = [
    "Srishtik/Qwen3-0.6B-linear-3-adapters-merged-2",
    "Srishtik/Qwen3-0.6B-svd-3-adapters-merged-2",
    "Srishtik/Qwen3-0.6B-ties-3-adapters-merged-2",
    "Srishtik/Qwen3-0.6B-dare-3-adapters-merged-2",
    "Srishtik/Qwen3-0.6B-bwsum-3-adapters-merged-2",
] + slerp_order_repos

specialist_repos = {
    "codealpaca_adapter": "Srishtik/qwen3-trained-on-code-alpaca-18k",
    "metamath_adapter"  : "Srishtik/qwen3-trained-on-metamath-15k",
    "dolly_adapter"     : "Srishtik/qwen3-trained-on-dolly-15k",
}

repos = {r: r for r in merged_repos}
repos.update(specialist_repos)

ORIGINAL_EM = {
    "Srishtik/Qwen3-0.6B-linear-3-adapters-merged-2": 0.1100,
    "Srishtik/Qwen3-0.6B-svd-3-adapters-merged-2": 0.1550,
    "Srishtik/Qwen3-0.6B-ties-3-adapters-merged-2": 0.0550,
    "Srishtik/Qwen3-0.6B-dare-3-adapters-merged-2": 0.0400,
    "Srishtik/Qwen3-0.6B-bwsum-3-adapters-merged-2": 0.1200,
    "Srishtik/Qwen3-0.6B-slerp-order-dolly-metamath-codealpaca-3-adapters-merged-2": 0.0100,
    "Srishtik/Qwen3-0.6B-slerp-order-dolly-codealpaca-metamath-3-adapters-merged-2": 0.0050,
    "Srishtik/Qwen3-0.6B-slerp-order-metamath-dolly-codealpaca-3-adapters-merged-2": 0.0100,
    "Srishtik/Qwen3-0.6B-slerp-order-metamath-codealpaca-dolly-3-adapters-merged-2": 0.0100,
    "Srishtik/Qwen3-0.6B-slerp-order-codealpaca-dolly-metamath-3-adapters-merged-2": 0.0050,
    "Srishtik/Qwen3-0.6B-slerp-order-codealpaca-metamath-dolly-3-adapters-merged-2": 0.0100,
    "Srishtik/qwen3-trained-on-code-alpaca-18k": 0.0850,
    "Srishtik/qwen3-trained-on-metamath-15k": 0.2200,
    "Srishtik/qwen3-trained-on-dolly-15k": 0.0250,
}
ORIGINAL_LEN_CAP_HITS = {
    "Srishtik/Qwen3-0.6B-linear-3-adapters-merged-2": 4,
    "Srishtik/Qwen3-0.6B-svd-3-adapters-merged-2": 4,
    "Srishtik/Qwen3-0.6B-ties-3-adapters-merged-2": 50,
    "Srishtik/Qwen3-0.6B-dare-3-adapters-merged-2": 6,
    "Srishtik/Qwen3-0.6B-bwsum-3-adapters-merged-2": 8,
    "Srishtik/Qwen3-0.6B-slerp-order-dolly-metamath-codealpaca-3-adapters-merged-2": 155,
    "Srishtik/Qwen3-0.6B-slerp-order-dolly-codealpaca-metamath-3-adapters-merged-2": 120,
    "Srishtik/Qwen3-0.6B-slerp-order-metamath-dolly-codealpaca-3-adapters-merged-2": 155,
    "Srishtik/Qwen3-0.6B-slerp-order-metamath-codealpaca-dolly-3-adapters-merged-2": 124,
    "Srishtik/Qwen3-0.6B-slerp-order-codealpaca-dolly-metamath-3-adapters-merged-2": 120,
    "Srishtik/Qwen3-0.6B-slerp-order-codealpaca-metamath-dolly-3-adapters-merged-2": 124,
    "Srishtik/qwen3-trained-on-code-alpaca-18k": 0,
    "Srishtik/qwen3-trained-on-metamath-15k": 22,
    "Srishtik/qwen3-trained-on-dolly-15k": 121,
}

try:
    with open("gsm8k_diagnostic_results.pkl", "rb") as f:
        _prior = pickle.load(f)
    for name, repo_id in repos.items():
        if name in _prior:
            ORIGINAL_EM[repo_id] = _prior[name]["diagnostic"]["exact_match"]
            ORIGINAL_LEN_CAP_HITS[repo_id] = _prior[name]["diagnostic"]["len_cap_hits"]
    print("Loaded original EM / len-cap-hit values from gsm8k_diagnostic_results.pkl")
except FileNotFoundError:
    print("gsm8k_diagnostic_results.pkl not found — using hardcoded ORIGINAL_EM fallback values")

gsm8k_diagnostic_results.pkl not found — using hardcoded ORIGINAL_EM fallback values


## Stage 1 — all 14 models at `max_new_tokens=512`, `repetition_penalty=1.0`

`temperature`/`top_p` untouched (greedy decoding throughout, same as the original notebook — no
sampling params were in play to begin with).

In [5]:
def run_ablation(repo_id, model_name, max_new_tokens, num_samples=200, batch_size=4):
    print(f"\n{'═'*60}")
    print(f"Ablation load: {repo_id}  (max_new_tokens={max_new_tokens})")
    print(f"{'═'*60}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = repo_id,
        max_seq_length = 1024,
        load_in_4bit   = True,
        dtype          = torch.float16,
    )
    FastLanguageModel.for_inference(model)

    diag = evaluate_gsm8k_diagnostic(
        model, tokenizer, model_name=f"{model_name}_ablation",
        num_samples=num_samples, batch_size=batch_size,
        max_new_tokens=max_new_tokens, repetition_penalty=1.0,
    )

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return diag


gsm8k_ablation_512 = {}
for name, repo_id in repos.items():
    gsm8k_ablation_512[name] = run_ablation(repo_id, name, max_new_tokens=512)

with open("gsm8k_ablation_512.pkl", "wb") as f:
    pickle.dump(gsm8k_ablation_512, f)

print(f"\nSaved gsm8k_ablation_512.pkl — {len(gsm8k_ablation_512)} models.")


════════════════════════════════════════════════════════════
Ablation load: Srishtik/Qwen3-0.6B-linear-3-adapters-merged-2  (max_new_tokens=512)
════════════════════════════════════════════════════════════
==((====))==  Unsloth 2026.7.5: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
[GSM8K-diagnostic] Evaluating: Srishtik/Qwen3-0.6B-linear-3-adapters-merged-2_ablation  (max_new_tokens=512, repetition_penalty=1.0)
────────────────────────────────────────────────────────────


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

...

  Exact Match: 0.5250  |  mean gen length: 130.9  |  len-cap hit on 3/200 (1.5%)
  path                       n  correct     acc
  hash                     197      105   0.533
  fallback_last_number       3        0   0.000

════════════════════════════════════════════════════════════
Ablation load: Srishtik/qwen3-trained-on-dolly-15k  (max_new_tokens=512)
════════════════════════════════════════════════════════════
==((====))==  Unsloth 2026.7.5: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
[GSM8K-diagnostic] Evaluating: dolly_adapter_ablation  (max_new_tokens=512, repetition_penalty=1.0)
────────────────────────────────────────────────────────────


dolly_adapter_ablation [gsm8k-diag]: 100%|██████████| 50/50 [23:19<00:00, 27.98s/it]


  Exact Match: 0.1600  |  mean gen length: 201.1  |  len-cap hit on 42/200 (21.0%)
  path                       n  correct     acc
  fallback_last_number     197       32   0.162
  no_number_found            2        0   0.000
  hash                       1        0   0.000

Saved gsm8k_ablation_512.pkl — 14 models.


## Stage 2 (conditional) — rerun at `max_new_tokens=768` for models still hitting the cap

Only models with a stage-1 cap-hit rate **> 10%** (>20/200 samples) get rerun. If none qualify,
this cell does nothing and stage 1 is the final answer for every model.

In [6]:
CAP_HIT_RATE_THRESHOLD = 0.10  # >10% still hitting max_new_tokens=512 triggers a 768 rerun

stage2_candidates = {
    name: repo_id for name, repo_id in repos.items()
    if gsm8k_ablation_512[name]["cap_hit_rate"] > CAP_HIT_RATE_THRESHOLD
}

print(f"{len(stage2_candidates)}/{len(repos)} models still above {CAP_HIT_RATE_THRESHOLD:.0%} cap-hit rate at 512 tokens:")
for name in stage2_candidates:
    r = gsm8k_ablation_512[name]
    print(f"  {name:<25} cap_hit_rate={r['cap_hit_rate']:.1%}  mean_gen_len={r['mean_gen_len']}")

gsm8k_ablation_768 = {}
for name, repo_id in stage2_candidates.items():
    gsm8k_ablation_768[name] = run_ablation(repo_id, name, max_new_tokens=768)

if gsm8k_ablation_768:
    with open("gsm8k_ablation_768.pkl", "wb") as f:
        pickle.dump(gsm8k_ablation_768, f)
    print(f"\nSaved gsm8k_ablation_768.pkl — {len(gsm8k_ablation_768)} models.")
else:
    print("\nNo model exceeded the cap-hit threshold at 512 tokens — stage 2 skipped entirely.")

7/14 models still above 10% cap-hit rate at 512 tokens:
  Srishtik/Qwen3-0.6B-slerp-order-dolly-metamath-codealpaca-3-adapters-merged-2 cap_hit_rate=37.5%  mean_gen_len=238.1
  Srishtik/Qwen3-0.6B-slerp-order-dolly-codealpaca-metamath-3-adapters-merged-2 cap_hit_rate=33.0%  mean_gen_len=211.6
  Srishtik/Qwen3-0.6B-slerp-order-metamath-dolly-codealpaca-3-adapters-merged-2 cap_hit_rate=37.5%  mean_gen_len=238.1
  Srishtik/Qwen3-0.6B-slerp-order-metamath-codealpaca-dolly-3-adapters-merged-2 cap_hit_rate=29.0%  mean_gen_len=208.2
  Srishtik/Qwen3-0.6B-slerp-order-codealpaca-dolly-metamath-3-adapters-merged-2 cap_hit_rate=33.0%  mean_gen_len=211.6
  Srishtik/Qwen3-0.6B-slerp-order-codealpaca-metamath-dolly-3-adapters-merged-2 cap_hit_rate=29.0%  mean_gen_len=208.2
  dolly_adapter             cap_hit_rate=21.0%  mean_gen_len=201.1

════════════════════════════════════════════════════════════
Ablation load: Srishtik/Qwen3-0.6B-slerp-order-dolly-metamath-codealpaca-3-adapters-merged-2  (max_ne

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
[GSM8K-diagnostic] Evaluating: Srishtik/Qwen3-0.6B-slerp-order-dolly-metamath-codealpaca-3-adapters-merged-2_ablation  (max_new_tokens=768, repetition_penalty=1.0)
────────────────────────────────────────────────────────────


Srishtik/Qwen3-0.6B-slerp-order-dolly-metamath-codealpaca-3-adapters-merged-2_ablation [gsm8k-diag]: 100%|██████████| 50/50 [31:43<00:00, 38.07s/it]


  Exact Match: 0.1800  |  mean gen length: 334.1  |  len-cap hit on 75/200 (37.5%)
  path                       n  correct     acc
  fallback_last_number     157       20   0.127
  hash                      43       16   0.372

════════════════════════════════════════════════════════════
Ablation load: Srishtik/Qwen3-0.6B-slerp-order-dolly-codealpaca-metamath-3-adapters-merged-2  (max_new_tokens=768)
════════════════════════════════════════════════════════════
==((====))==  Unsloth 2026.7.5: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

  Exact Match: 0.2250  |  mean gen length: 282.4  |  len-cap hit on 58/200 (29.0%)
  path                       n  correct     acc
  fallback_last_number     143       21   0.147
  hash                      57       24   0.421

════════════════════════════════════════════════════════════
Ablation load: Srishtik/qwen3-trained-on-dolly-15k  (max_new_tokens=768)
════════════════════════════════════════════════════════════
==((====))==  Unsloth 2026.7.5: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


────────────────────────────────────────────────────────────
[GSM8K-diagnostic] Evaluating: dolly_adapter_ablation  (max_new_tokens=768, repetition_penalty=1.0)
────────────────────────────────────────────────────────────


dolly_adapter_ablation [gsm8k-diag]: 100%|██████████| 50/50 [31:53<00:00, 38.27s/it]


  Exact Match: 0.1600  |  mean gen length: 254.8  |  len-cap hit on 42/200 (21.0%)
  path                       n  correct     acc
  fallback_last_number     197       32   0.162
  no_number_found            2        0   0.000
  hash                       1        0   0.000

Saved gsm8k_ablation_768.pkl — 7 models.


## Final comparison + verdict per model

For each model: original (320/1.3) EM, best available ablation EM (768 result if it was rerun,
else 512), mean generated length, cap-hit rate, and a verdict —
`budget-limited` (cap-hit rate dropped a lot and EM rose) vs. `weight-limited` (cap-hit rate was
already low, or stayed low, and EM is still poor).

In [7]:
print(f"{'model':<25} {'orig_EM':>8} {'final_EM':>9} {'delta':>8} {'gen_len':>8} {'cap_hit%':>9}  verdict")
print("─" * 100)

rows = []
for name, repo_id in repos.items():
    final = gsm8k_ablation_768.get(name, gsm8k_ablation_512[name])
    orig_em   = ORIGINAL_EM[repo_id]
    final_em  = final["exact_match"]
    delta     = final_em - orig_em
    cap_rate  = final["cap_hit_rate"]
    gen_len   = final["mean_gen_len"]

    if cap_rate <= 0.05 and delta < 0.03:
        verdict = "weight-limited"
    elif delta >= 0.03:
        verdict = "budget-limited"
    else:
        verdict = "inconclusive"

    rows.append((name, orig_em, final_em, delta, gen_len, cap_rate, verdict))

for name, orig_em, final_em, delta, gen_len, cap_rate, verdict in sorted(rows, key=lambda r: -r[3]):
    print(f"{name:<25} {orig_em:>8.4f} {final_em:>9.4f} {delta:>+8.4f} {gen_len:>8.1f} {cap_rate:>8.1%}  {verdict}")

with open("gsm8k_ablation_final_comparison.pkl", "wb") as f:
    pickle.dump(rows, f)

model                      orig_EM  final_EM    delta  gen_len  cap_hit%  verdict
────────────────────────────────────────────────────────────────────────────────────────────────────
metamath_adapter            0.2200    0.5250  +0.3050    130.9     1.5%  budget-limited
Srishtik/Qwen3-0.6B-ties-3-adapters-merged-2   0.0550    0.3200  +0.2650    111.3     4.0%  budget-limited
Srishtik/Qwen3-0.6B-linear-3-adapters-merged-2   0.1100    0.3600  +0.2500    108.3     2.0%  budget-limited
Srishtik/Qwen3-0.6B-svd-3-adapters-merged-2   0.1550    0.3950  +0.2400    111.0     1.5%  budget-limited
Srishtik/Qwen3-0.6B-slerp-order-metamath-codealpaca-dolly-3-adapters-merged-2   0.0100    0.2250  +0.2150    282.4    29.0%  budget-limited
Srishtik/Qwen3-0.6B-slerp-order-codealpaca-metamath-dolly-3-adapters-merged-2   0.0100    0.2250  +0.2150    282.4    29.0%  budget-limited
Srishtik/Qwen3-0.6B-bwsum-3-adapters-merged-2   0.1200    0.3100  +0.1900    107.5     1.5%  budget-limited
Srishtik/Qwen3-0.6B